In [1]:
import os
import sys
from pathlib import Path
import xarray as xr
import pandas as pd
import numpy as np
import coiled
from dask.distributed import Client, LocalCluster
import dask
import zarr
import gcsfs
import json
import yaml
import re
import logging
from IPython.core.magic import register_cell_magic
@register_cell_magic
def comment(line, cell):
    # This magic does nothing, effectively ignoring the cell's content
    pass
# Add app to path
sys.path.insert(0, "/home/stefan/CRS/CRS.ZarrPipelines/app")
from utils import HazardConfig, get_thresholds, score
from domain import scoring

dask.config.set({"array.rechunk.method": "p2p"})
print(os.getcwd())

/home/stefan/CRS/CRS.ZarrPipelines/.venv/lib/python3.9/site-packages/google/oauth2/__init__.py:40: FutureWarning: You are using a Python version 3.9 past its end of life. Google will update google-auth with critical bug fixes on a best-effort basis, but not with any other fixes or features. Please upgrade your Python version, and then update google-auth.
  warnings.warn(eol_message.format("3.9"), FutureWarning)
/home/stefan/CRS/CRS.ZarrPipelines/.venv/lib/python3.9/site-packages/google/auth/__init__.py:54: FutureWarning: You are using a Python version 3.9 past its end of life. Google will update google-auth with critical bug fixes on a best-effort basis, but not with any other fixes or features. Please upgrade your Python version, and then update google-auth.
  warnings.warn(eol_message.format("3.9"), FutureWarning)


ImportError: cannot import name 'HazardConfig' from 'utils' (/home/stefan/CRS/CRS.ZarrPipelines/app/utils/__init__.py)

## Data Paths

In [2]:
INPUT_PATH = 'gs://crs_climate_data_public/PHYSICAL_HAZARDS_WORKING_FOLDER/WF/zarrs/processed_unscored/WF.zarr/'
OUTPUT_PATH_1_to_5 = 'gs://crs_climate_data_public/production_test/hazard_scores_1_to_5/WF.zarr'
OUTPUT_PATH_1_to_3 = 'gs://crs_climate_data_public/production_test/hazard_scores_1_to_3/WF.zarr'
OUTPUT_PATH_1_to_10 = 'gs://crs_climate_data_public/production_test/hazard_scores_1_to_10/WF.zarr'
OUTPUT_PATH_1_to_100_norm = 'gs://crs_climate_data_public/production_test/hazard_scores_1_to_100_norm/WF.zarr'

# Spin up Coiled Cluster

In [3]:
%%comment
# Initialize cluster
cluster = coiled.Cluster(
    n_workers=15,
    scheduler_vm_types="e2-standard-8",
    worker_vm_types="e2-standard-4",
    region="europe-west8",
    name="wonder",
    shutdown_on_close=False,
    idle_timeout="2 hours"
)
client = Client(cluster)
cluster.adapt(minimum=15, maximum=20)
logging.info(f"Cluster setup complete: {client}")

## Hazard Data

In [4]:
ds = xr.open_zarr(INPUT_PATH)
ds

<xarray.Dataset> Size: 14GB
Dimensions:      (lat: 17520, lon: 43200, time: 4, scenario: 2, model: 1,
                  metric: 1)
Coordinates:
  * lat          (lat) float64 140kB 90.0 89.99 89.98 ... -55.98 -55.99 -56.0
  * lon          (lon) float64 346kB -180.0 -180.0 -180.0 ... 180.0 180.0 180.0
  * metric       (metric) <U3 12B 'p95'
  * model        (model) <U9 36B 'GFDL-ESM4'
  * scenario     (scenario) <U5 40B 'RCP45' 'RCP85'
  * time         (time) <U2 32B 'Cc' 'St' 'Mt' 'Lt'
Data variables:
    burnability  (lat, lon) float16 2GB dask.array<chunksize=(1024, 1024), meta=np.ndarray>
    fwi          (time, scenario, model, metric, lat, lon) float16 12GB dask.array<chunksize=(1, 1, 1, 1, 1024, 1024), meta=np.ndarray>

# Step 1 : Create Unscored data according to config instructions

In [6]:
# Initialize config
config = HazardConfig()

Found config at: /home/stefan/CRS/CRS.ZarrPipelines/app/config/config.yaml


In [7]:
# Get all thresholds for LS
ls_all = config.get_all_thresholds('WF')
print("All WF thresholds:")
print(ls_all)

All WF thresholds:
{'hazard': 'Wildfire', 'code': 'WF', 'metrics': {'p95': {'scoring.5_point': {'thresholds': [0, 1e-06, 5.2, 11.2, 21.3, 50, inf], 'scores': [0, 1, 2, 3, 4, 5]}, 'scoring.10_point': {'thresholds': [0, 1e-06, 2.6, 5.2, 8.2, 11.2, 16.25, 21.3, 35.65, 43, 50, inf], 'scores': [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10]}}}}


# 1-5

In [8]:
thresh_wf_5, scores_wf_5 = config.get_thresholds('WF', '5', threshold_type='p95')
thresh_wf_5

[0, 1e-06, 5.2, 11.2, 21.3, 50, inf]

# 1-10

In [17]:
thresh_wf_10, scores_wf_10 = config.get_thresholds('WF', '10', threshold_type='p95')
thresh_wf_10

[0, 1e-06, 2.6, 5.2, 8.2, 11.2, 16.25, 21.3, 35.65, 43, 50, inf]

# Step 2: Score Data according to bins with digitize

In [13]:
wf = ds['burnability']*ds['fwi']
wf

<xarray.DataArray (lat: 17520, lon: 43200, time: 4, scenario: 2, model: 1,
                   metric: 1)> Size: 12GB
dask.array<mul, shape=(17520, 43200, 4, 2, 1, 1), dtype=float16, chunksize=(1024, 1024, 1, 1, 1, 1), chunktype=numpy.ndarray>
Coordinates:
  * lat       (lat) float64 140kB 90.0 89.99 89.98 89.97 ... -55.98 -55.99 -56.0
  * lon       (lon) float64 346kB -180.0 -180.0 -180.0 ... 180.0 180.0 180.0
  * metric    (metric) <U3 12B 'p95'
  * model     (model) <U9 36B 'GFDL-ESM4'
  * scenario  (scenario) <U5 40B 'RCP45' 'RCP85'
  * time      (time) <U2 32B 'Cc' 'St' 'Mt' 'Lt'

## 1-5

In [14]:
scored_final_5 = scoring.score_zarr(
    wf,
    thresh_wf_5,
    scores_wf_5,
    metric = 'p95'
)
scored_final_5

Scoring array with shape (17520, 43200, 4, 2, 1, 1)
Thresholds: [0, 1e-06, 5.2, 11.2, 21.3, 50, inf]
Scores: [0, 1, 2, 3, 4, 5]
Setting metric coordinate to: p95


<xarray.Dataset> Size: 48GB
Dimensions:   (lat: 17520, lon: 43200, model: 1, scenario: 2, time: 4)
Coordinates:
  * lat       (lat) float64 140kB 90.0 89.99 89.98 89.97 ... -55.98 -55.99 -56.0
  * lon       (lon) float64 346kB -180.0 -180.0 -180.0 ... 180.0 180.0 180.0
  * model     (model) <U9 36B 'GFDL-ESM4'
  * scenario  (scenario) <U5 40B 'RCP45' 'RCP85'
  * time      (time) <U2 32B 'Cc' 'St' 'Mt' 'Lt'
    metric    <U3 12B 'p95'
Data variables:
    score     (lat, lon, time, scenario, model) float64 48GB dask.array<chunksize=(1024, 1024, 1, 1, 1), meta=np.ndarray>

# 1 - 10

In [18]:
scored_final_10 = scoring.score_zarr(
    wf,
    thresh_wf_10,
    scores_wf_10,
    metric = 'p95'
)
scored_final_10

Scoring array with shape (17520, 43200, 4, 2, 1, 1)
Thresholds: [0, 1e-06, 2.6, 5.2, 8.2, 11.2, 16.25, 21.3, 35.65, 43, 50, inf]
Scores: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
Setting metric coordinate to: p95


<xarray.Dataset> Size: 48GB
Dimensions:   (lat: 17520, lon: 43200, model: 1, scenario: 2, time: 4)
Coordinates:
  * lat       (lat) float64 140kB 90.0 89.99 89.98 89.97 ... -55.98 -55.99 -56.0
  * lon       (lon) float64 346kB -180.0 -180.0 -180.0 ... 180.0 180.0 180.0
  * model     (model) <U9 36B 'GFDL-ESM4'
  * scenario  (scenario) <U5 40B 'RCP45' 'RCP85'
  * time      (time) <U2 32B 'Cc' 'St' 'Mt' 'Lt'
    metric    <U3 12B 'p95'
Data variables:
    score     (lat, lon, time, scenario, model) float64 48GB dask.array<chunksize=(1024, 1024, 1, 1, 1), meta=np.ndarray>

# Step 3: GADM Aggregations with xvec?
## https://earthmover.io/blog/vector-datacube-pt1/

## GADM data